# 04 - Multivariate Feature Selection

This notebook evaluates the probabilistic classifiers after applying a
multivariate filter feature-selection strategy.

Correlation-based Feature Selection (CFS) is used because it evaluates a subset
of predictors jointly rather than scoring every variable independently.

A good CFS subset should contain predictors that:

- are strongly associated with the target;
- provide complementary information;
- are not excessively redundant with each other.

The feature-selection procedure is fitted independently inside each
cross-validation training fold to prevent information leakage.

In [5]:
import pandas as pd
import numpy as np

from scipy.special import logsumexp

from sklearn.base import (
    BaseEstimator,
    TransformerMixin,
    ClassifierMixin,
    clone
)

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    KBinsDiscretizer
)

from sklearn.pipeline import Pipeline

from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    mutual_info_score
)

from sklearn.linear_model import LogisticRegression

from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 1. Dataset Preparation

The same dataset preparation used in the previous PA2 experiments is retained.

The identifier and constant attributes are removed, while `Attrition` is
encoded as a binary target:

- `No` -> 0
- `Yes` -> 1

In [6]:
DATA_PATH = "../../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"

df = pd.read_csv(DATA_PATH)

columns_to_remove = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

df_model = df.drop(
    columns=columns_to_remove
)

X = df_model.drop(
    columns=["Attrition"]
)

y = df_model["Attrition"].map({
    "No": 0,
    "Yes": 1
})

print(
    "Predictor matrix shape:",
    X.shape
)

print(
    "Target shape:",
    y.shape
)

Predictor matrix shape: (1470, 30)
Target shape: (1470,)


## 2. Predictor Types

The original distinction between continuous, ordinal, and nominal predictors
is retained.

For CFS association calculations, predictors require a common discrete
representation.

Continuous variables will therefore be discretized only for the feature-
selection criterion.

This temporary representation is not the representation subsequently used by
the classifiers.

In [7]:
ordinal_features = [
    "Education",
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "JobLevel",
    "JobSatisfaction",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "WorkLifeBalance"
]

numerical_features = (
    X.select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    X.select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

continuous_features = [
    feature
    for feature in numerical_features
    if feature not in ordinal_features
]

discrete_features = (
    ordinal_features
    + categorical_features
)

print(
    "Continuous:",
    len(continuous_features)
)

print(
    "Discrete/ordinal:",
    len(discrete_features)
)

print(
    "Total:",
    len(continuous_features)
    + len(discrete_features)
)

Continuous: 14
Discrete/ordinal: 16
Total: 30


## 3. Association Measure

CFS requires measures of:

- predictor-target association;
- predictor-predictor association.

Because the dataset contains mixed predictor types, a common discrete
representation is used for the CFS criterion.

Association is measured using Symmetrical Uncertainty (SU), an
information-theoretic normalized measure derived from Mutual Information.

SU ranges from 0 to 1:

- values close to 0 indicate weak association;
- values close to 1 indicate strong association.

This allows feature relevance and redundancy to be expressed on the same
scale.

In [8]:
def entropy(values):

    values = np.asarray(
        values
    )

    _, counts = np.unique(
        values,
        return_counts=True
    )

    probabilities = (
        counts / counts.sum()
    )

    return -np.sum(
        probabilities
        * np.log2(probabilities)
    )


def symmetrical_uncertainty(
    x,
    y
):

    hx = entropy(x)
    hy = entropy(y)

    if hx + hy == 0:
        return 0.0

    mi = mutual_info_score(
        x,
        y
    )

    # mutual_info_score uses natural logarithms.
    # Convert MI to bits so that it is expressed
    # on the same scale as the entropy function.
    mi_bits = (
        mi / np.log(2)
    )

    return (
        2.0
        * mi_bits
        / (hx + hy)
    )

## 4. CFS Representation

For the CFS criterion:

- Continuous variables are discretized into quantile-based bins.
- Ordinal variables retain their original integer categories.
- Nominal variables are converted to integer category codes.

Discretization and category encoding are fitted only on the training data of
each cross-validation fold.

The resulting representation is used only by the feature-selection algorithm.

## 5. Correlation-based Feature Selection

For a subset containing \(k\) variables, the CFS merit criterion combines:

- the average association between predictors and the target;
- the average association among the predictors themselves.

A subset receives a high merit when its variables are individually relevant
to the target but not strongly redundant with each other.

A greedy forward search is used.

Starting from an empty subset, the variable producing the greatest increase in
CFS merit is added at each iteration.

The search evaluates subsets containing up to 15 original predictors and
retains the best subset encountered.

In [9]:
class CFSSelector(
    BaseEstimator,
    TransformerMixin
):

    def __init__(
        self,
        continuous_features,
        discrete_features,
        max_features=15,
        n_bins=5
    ):
        self.continuous_features = continuous_features
        self.discrete_features = discrete_features
        self.max_features = max_features
        self.n_bins = n_bins

    def _fit_discrete_representation(
        self,
        X
    ):

        X_encoded = pd.DataFrame(
            index=X.index
        )

        self.bin_edges_ = {}
        self.category_maps_ = {}

        # ----------------------
        # Continuous predictors
        # ----------------------
        for feature in self.continuous_features:

            values = (
                X[feature]
                .astype(float)
                .to_numpy()
            )

            quantiles = np.linspace(
                0,
                1,
                self.n_bins + 1
            )

            edges = np.unique(
                np.quantile(
                    values,
                    quantiles
                )
            )

            # A constant or nearly constant
            # variable may not create enough bins.
            if len(edges) <= 2:

                encoded = np.zeros(
                    len(values),
                    dtype=int
                )

                self.bin_edges_[
                    feature
                ] = None

            else:

                internal_edges = (
                    edges[1:-1]
                )

                encoded = np.digitize(
                    values,
                    internal_edges
                )

                self.bin_edges_[
                    feature
                ] = internal_edges

            X_encoded[
                feature
            ] = encoded

        # ----------------------
        # Discrete predictors
        # ----------------------
        for feature in self.discrete_features:

            values = (
                X[feature]
                .astype(str)
            )

            categories = (
                values
                .unique()
                .tolist()
            )

            category_map = {
                category: index
                for index, category
                in enumerate(categories)
            }

            self.category_maps_[
                feature
            ] = category_map

            X_encoded[
                feature
            ] = (
                values
                .map(category_map)
                .astype(int)
            )

        return X_encoded

    def _subset_merit(
        self,
        subset,
        relevance,
        redundancy
    ):

        k = len(subset)

        if k == 0:
            return 0.0

        average_relevance = np.mean([
            relevance[feature]
            for feature in subset
        ])

        if k == 1:

            average_redundancy = 0.0

        else:

            redundancy_values = []

            for i in range(k):

                for j in range(
                    i + 1,
                    k
                ):

                    feature_i = subset[i]
                    feature_j = subset[j]

                    key = tuple(
                        sorted([
                            feature_i,
                            feature_j
                        ])
                    )

                    redundancy_values.append(
                        redundancy[key]
                    )

            average_redundancy = np.mean(
                redundancy_values
            )

        denominator = np.sqrt(
            k
            + k * (k - 1)
            * average_redundancy
        )

        if denominator == 0:
            return 0.0

        return (
            k
            * average_relevance
            / denominator
        )

    def fit(
        self,
        X,
        y
    ):

        X = X.copy()
        y = np.asarray(y)

        X_encoded = (
            self._fit_discrete_representation(
                X
            )
        )

        features = (
            X.columns
            .tolist()
        )

        # ----------------------
        # Feature-target relevance
        # ----------------------
        relevance = {}

        for feature in features:

            relevance[feature] = (
                symmetrical_uncertainty(
                    X_encoded[feature],
                    y
                )
            )

        # ----------------------
        # Feature-feature redundancy
        # ----------------------
        redundancy = {}

        for i, feature_i in enumerate(
            features
        ):

            for feature_j in features[
                i + 1:
            ]:

                key = tuple(
                    sorted([
                        feature_i,
                        feature_j
                    ])
                )

                redundancy[key] = (
                    symmetrical_uncertainty(
                        X_encoded[feature_i],
                        X_encoded[feature_j]
                    )
                )

        # ----------------------
        # Greedy forward search
        # ----------------------
        selected = []
        remaining = features.copy()

        best_subset = []
        best_merit = -np.inf

        search_history = []

        while (
            remaining
            and len(selected)
            < self.max_features
        ):

            candidate_results = []

            for feature in remaining:

                candidate_subset = (
                    selected
                    + [feature]
                )

                merit = self._subset_merit(
                    candidate_subset,
                    relevance,
                    redundancy
                )

                candidate_results.append(
                    (
                        feature,
                        merit
                    )
                )

            best_feature, current_merit = max(
                candidate_results,
                key=lambda item: item[1]
            )

            selected.append(
                best_feature
            )

            remaining.remove(
                best_feature
            )

            search_history.append({
                "Step": len(selected),
                "AddedFeature": best_feature,
                "Merit": current_merit
            })

            if current_merit > best_merit:

                best_merit = (
                    current_merit
                )

                best_subset = (
                    selected.copy()
                )

        self.selected_features_ = (
            best_subset
        )

        self.best_merit_ = (
            best_merit
        )

        self.relevance_ = relevance

        self.search_history_ = (
            pd.DataFrame(
                search_history
            )
        )

        return self

    def transform(
        self,
        X
    ):

        return X[
            self.selected_features_
        ].copy()

    def get_feature_names_out(
        self,
        input_features=None
    ):

        return np.array(
            self.selected_features_
        )

## 6. Descriptive Full-Data Selection

CFS is fitted once on the complete dataset only to inspect the subset and the
search process.

This fit is descriptive and is not used to estimate predictive performance.

During cross-validation, CFS is refitted independently using only each
training fold.

In [10]:
descriptive_cfs = CFSSelector(
    continuous_features=continuous_features,
    discrete_features=discrete_features,
    max_features=15,
    n_bins=5
)

descriptive_cfs.fit(
    X,
    y
)

print(
    "Selected variables:",
    len(
        descriptive_cfs
        .selected_features_
    )
)

print()

print(
    descriptive_cfs
    .selected_features_
)

print(
    "\nBest CFS merit:",
    round(
        descriptive_cfs
        .best_merit_,
        4
    )
)

Selected variables: 9

['OverTime', 'JobLevel', 'StockOptionLevel', 'YearsAtCompany', 'Age', 'BusinessTravel', 'JobRole', 'MaritalStatus', 'JobInvolvement']

Best CFS merit: 0.0642


In [11]:
display(
    descriptive_cfs
    .search_history_
    .round(4)
)

,Step,AddedFeature,Merit
0,1,OverTime,0.0534
1,2,JobLevel,0.0574
2,3,StockOptionLevel,0.0621
3,4,YearsAtCompany,0.0635
4,5,Age,0.0628
5,6,BusinessTravel,0.0629
6,7,JobRole,0.0633
7,8,MaritalStatus,0.0637
8,9,JobInvolvement,0.0642
9,10,YearsWithCurrManager,0.0642


In [12]:
relevance_df = (
    pd.DataFrame({
        "Feature":
            descriptive_cfs
            .relevance_
            .keys(),

        "TargetAssociation":
            descriptive_cfs
            .relevance_
            .values()
    })
    .sort_values(
        "TargetAssociation",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

display(
    relevance_df.round(4)
)

,Feature,TargetAssociation
0,OverTime,0.0534
1,JobLevel,0.0279
2,StockOptionLevel,0.0266
3,JobRole,0.0245
4,YearsAtCompany,0.0224
5,MonthlyIncome,0.0213
6,TotalWorkingYears,0.0210
7,YearsWithCurrManager,0.0205
8,MaritalStatus,0.0200
9,Age,0.0175


## 7. Mixed Naive Bayes

The same Mixed Naive Bayes formulation used in the previous probabilistic
experiments is retained.

The classifier automatically uses only the continuous and discrete predictors
retained by CFS in each training fold.

In [13]:
class MixedNaiveBayes(
    BaseEstimator,
    ClassifierMixin
):
    def __init__(
        self,
        continuous_features,
        discrete_features,
        alpha=1.0,
        var_smoothing=1e-9
    ):
        self.continuous_features = continuous_features
        self.discrete_features = discrete_features
        self.alpha = alpha
        self.var_smoothing = var_smoothing

    def fit(self, X, y):

        X = X.copy()
        y = np.asarray(y)

        self.classes_ = np.unique(y)

        selected_columns = (
            X.columns.tolist()
        )

        self.active_continuous_features_ = [
            feature
            for feature in self.continuous_features
            if feature in selected_columns
        ]

        self.active_discrete_features_ = [
            feature
            for feature in self.discrete_features
            if feature in selected_columns
        ]

        self.class_prior_ = {}
        self.gaussian_parameters_ = {}
        self.discrete_probabilities_ = {}
        self.discrete_default_probabilities_ = {}

        n_samples = len(y)

        for cls in self.classes_:

            class_mask = (
                y == cls
            )

            X_cls = X.loc[
                class_mask
            ]

            class_count = (
                class_mask.sum()
            )

            self.class_prior_[cls] = (
                class_count
                / n_samples
            )

            self.gaussian_parameters_[cls] = {}

            for feature in (
                self.active_continuous_features_
            ):

                values = (
                    X_cls[feature]
                    .astype(float)
                    .to_numpy()
                )

                mean = (
                    values.mean()
                )

                variance = (
                    values.var()
                    + self.var_smoothing
                )

                self.gaussian_parameters_[
                    cls
                ][feature] = (
                    mean,
                    variance
                )

            self.discrete_probabilities_[cls] = {}
            self.discrete_default_probabilities_[cls] = {}

            for feature in (
                self.active_discrete_features_
            ):

                all_categories = (
                    X[feature]
                    .astype(str)
                    .unique()
                )

                class_values = (
                    X_cls[feature]
                    .astype(str)
                )

                counts = (
                    class_values
                    .value_counts()
                )

                n_categories = (
                    len(all_categories)
                    + 1
                )

                denominator = (
                    class_count
                    + self.alpha
                    * n_categories
                )

                probabilities = {}

                for category in all_categories:

                    probabilities[
                        category
                    ] = (
                        counts.get(
                            category,
                            0
                        )
                        + self.alpha
                    ) / denominator

                self.discrete_probabilities_[
                    cls
                ][feature] = (
                    probabilities
                )

                self.discrete_default_probabilities_[
                    cls
                ][feature] = (
                    self.alpha
                    / denominator
                )

        return self

    def _joint_log_likelihood(
        self,
        X
    ):

        X = X.copy()

        joint_log_likelihood = []

        for cls in self.classes_:

            log_probability = np.full(
                X.shape[0],
                np.log(
                    self.class_prior_[
                        cls
                    ]
                ),
                dtype=float
            )

            for feature in (
                self.active_continuous_features_
            ):

                mean, variance = (
                    self.gaussian_parameters_[
                        cls
                    ][feature]
                )

                values = (
                    X[feature]
                    .astype(float)
                    .to_numpy()
                )

                log_probability += (
                    -0.5
                    * np.log(
                        2
                        * np.pi
                        * variance
                    )
                    - (
                        (values - mean) ** 2
                        / (2 * variance)
                    )
                )

            for feature in (
                self.active_discrete_features_
            ):

                probabilities = (
                    self.discrete_probabilities_[
                        cls
                    ][feature]
                )

                default_probability = (
                    self.discrete_default_probabilities_[
                        cls
                    ][feature]
                )

                values = (
                    X[feature]
                    .astype(str)
                    .to_numpy()
                )

                feature_probabilities = np.array([
                    probabilities.get(
                        value,
                        default_probability
                    )
                    for value in values
                ])

                log_probability += np.log(
                    feature_probabilities
                )

            joint_log_likelihood.append(
                log_probability
            )

        return np.column_stack(
            joint_log_likelihood
        )

    def predict_proba(
        self,
        X
    ):

        joint_log_likelihood = (
            self._joint_log_likelihood(
                X
            )
        )

        log_normalizer = (
            logsumexp(
                joint_log_likelihood,
                axis=1,
                keepdims=True
            )
        )

        return np.exp(
            joint_log_likelihood
            - log_normalizer
        )

    def predict(
        self,
        X
    ):

        probabilities = (
            self.predict_proba(
                X
            )
        )

        return self.classes_[
            np.argmax(
                probabilities,
                axis=1
            )
        ]

In [14]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [15]:
def multiclass_brier_score(
    y_true,
    probabilities,
    classes
):

    y_true = np.asarray(
        y_true
    )

    indicator_matrix = np.zeros_like(
        probabilities,
        dtype=float
    )

    for index, cls in enumerate(
        classes
    ):

        indicator_matrix[
            :,
            index
        ] = (
            y_true == cls
        ).astype(float)

    squared_error = (
        probabilities
        - indicator_matrix
    ) ** 2

    return np.mean(
        squared_error.sum(
            axis=1
        )
    )

In [16]:
def create_preprocessors(
    selected_features
):

    selected_continuous = [
        feature
        for feature in continuous_features
        if feature in selected_features
    ]

    selected_ordinal = [
        feature
        for feature in ordinal_features
        if feature in selected_features
    ]

    selected_categorical = [
        feature
        for feature in categorical_features
        if feature in selected_features
    ]

    selected_numeric = (
        selected_continuous
        + selected_ordinal
    )

    logistic_preprocessor = (
        ColumnTransformer(
            transformers=[
                (
                    "num",
                    StandardScaler(),
                    selected_numeric
                ),
                (
                    "cat",
                    OneHotEncoder(
                        drop="first",
                        handle_unknown="ignore",
                        sparse_output=False
                    ),
                    selected_categorical
                )
            ]
        )
    )

    discriminant_preprocessor = (
        ColumnTransformer(
            transformers=[
                (
                    "num",
                    StandardScaler(),
                    selected_numeric
                ),
                (
                    "cat",
                    OrdinalEncoder(
                        handle_unknown="use_encoded_value",
                        unknown_value=-1
                    ),
                    selected_categorical
                )
            ]
        )
    )

    return (
        logistic_preprocessor,
        discriminant_preprocessor,
        selected_continuous,
        selected_ordinal,
        selected_categorical
    )

In [17]:
def evaluate_fold_model(
    name,
    model,
    X_train,
    X_test,
    y_train,
    y_test,
    fold
):

    model.fit(
        X_train,
        y_train
    )

    y_pred = (
        model.predict(
            X_test
        )
    )

    probabilities = (
        model.predict_proba(
            X_test
        )
    )

    classes = (
        model.classes_
    )

    positive_index = np.where(
        classes == 1
    )[0][0]

    positive_probability = (
        probabilities[
            :,
            positive_index
        ]
    )

    return {
        "Model": name,
        "Fold": fold,

        "Accuracy":
            accuracy_score(
                y_test,
                y_pred
            ),

        "Precision":
            precision_score(
                y_test,
                y_pred,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_test,
                y_pred
            ),

        "Specificity":
            recall_score(
                y_test,
                y_pred,
                pos_label=0
            ),

        "F1":
            f1_score(
                y_test,
                y_pred,
                zero_division=0
            ),

        "AUC":
            roc_auc_score(
                y_test,
                positive_probability
            ),

        "Brier":
            multiclass_brier_score(
                y_test,
                probabilities,
                classes
            )
    }

## 8. Multivariate Feature-Selection Experiment

For every stratified cross-validation fold:

1. CFS is fitted only on the training partition.
2. A multivariate subset of original predictors is selected.
3. Model-specific preprocessing is constructed for that subset.
4. The four probabilistic classifiers are trained.
5. Performance is measured on the untouched validation partition.

The selected subsets are also stored to analyse feature-selection stability.

In [18]:
fold_results = []
selection_records = []

for fold, (
    train_index,
    test_index
) in enumerate(
    cv.split(X, y),
    start=1
):

    print(
        f"Fold {fold}/10"
    )

    X_train = (
        X.iloc[
            train_index
        ].copy()
    )

    X_test = (
        X.iloc[
            test_index
        ].copy()
    )

    y_train = (
        y.iloc[
            train_index
        ].copy()
    )

    y_test = (
        y.iloc[
            test_index
        ].copy()
    )

    selector = CFSSelector(
        continuous_features=
            continuous_features,

        discrete_features=
            discrete_features,

        max_features=15,
        n_bins=5
    )

    selector.fit(
        X_train,
        y_train
    )

    selected_features = (
        selector
        .selected_features_
    )

    selection_records.append({
        "Fold": fold,

        "SelectedFeatures":
            selected_features,

        "NumberSelected":
            len(
                selected_features
            ),

        "CFS_Merit":
            selector.best_merit_
    })

    X_train_selected = (
        selector.transform(
            X_train
        )
    )

    X_test_selected = (
        selector.transform(
            X_test
        )
    )

    (
        logistic_preprocessor,
        discriminant_preprocessor,
        selected_continuous,
        selected_ordinal,
        selected_categorical
    ) = create_preprocessors(
        selected_features
    )

    selected_discrete = (
        selected_ordinal
        + selected_categorical
    )

    # Mixed Naive Bayes

    mixed_nb = MixedNaiveBayes(
        continuous_features=
            selected_continuous,

        discrete_features=
            selected_discrete,

        alpha=1.0
    )

    fold_results.append(
        evaluate_fold_model(
            "Mixed Naive Bayes",
            mixed_nb,
            X_train_selected,
            X_test_selected,
            y_train,
            y_test,
            fold
        )
    )

    # Logistic Regression

    logistic_model = Pipeline(
        steps=[
            (
                "preprocessing",
                clone(
                    logistic_preprocessor
                )
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=42
                )
            )
        ]
    )

    fold_results.append(
        evaluate_fold_model(
            "Logistic Regression",
            logistic_model,
            X_train_selected,
            X_test_selected,
            y_train,
            y_test,
            fold
        )
    )

    # LDA

    lda_model = Pipeline(
        steps=[
            (
                "preprocessing",
                clone(
                    discriminant_preprocessor
                )
            ),
            (
                "classifier",
                LinearDiscriminantAnalysis()
            )
        ]
    )

    fold_results.append(
        evaluate_fold_model(
            "LDA",
            lda_model,
            X_train_selected,
            X_test_selected,
            y_train,
            y_test,
            fold
        )
    )

    # QDA

    qda_model = Pipeline(
        steps=[
            (
                "preprocessing",
                clone(
                    discriminant_preprocessor
                )
            ),
            (
                "classifier",
                QuadraticDiscriminantAnalysis()
            )
        ]
    )

    fold_results.append(
        evaluate_fold_model(
            "QDA",
            qda_model,
            X_train_selected,
            X_test_selected,
            y_train,
            y_test,
            fold
        )
    )

print(
    "\nEvaluation completed."
)

Fold 1/10
Fold 2/10
Fold 3/10
Fold 4/10
Fold 5/10
Fold 6/10
Fold 7/10
Fold 8/10
Fold 9/10
Fold 10/10

Evaluation completed.


In [19]:
multivariate_fold_results = (
    pd.DataFrame(
        fold_results
    )
)

display(
    multivariate_fold_results
    .round(4)
)

,Model,Fold,Accuracy,Precision,Recall,Specificity,F1,AUC,Brier
0,Mixed Naive Bayes,1,0.8435,0.5455,0.2500,0.9593,0.3429,0.7586,0.2389
1,Logistic Regression,1,0.8571,0.7143,0.2083,0.9837,0.3226,0.6767,0.2589
2,LDA,1,0.8435,0.5556,0.2083,0.9675,0.3030,0.6855,0.2637
3,QDA,1,0.8163,0.3846,0.2083,0.9350,0.2703,0.6282,0.2738
4,Mixed Naive Bayes,2,0.8299,0.4737,0.3750,0.9187,0.4186,0.7409,0.2606
5,Logistic Regression,2,0.8503,0.6000,0.2500,0.9675,0.3529,0.8001,0.2228
6,LDA,2,0.8571,0.6667,0.2500,0.9756,0.3636,0.8042,0.2259
7,QDA,2,0.8367,0.5000,0.3750,0.9268,0.4286,0.7934,0.2363
8,Mixed Naive Bayes,3,0.7823,0.3333,0.3333,0.8699,0.3333,0.6701,0.2983
9,Logistic Regression,3,0.8163,0.2000,0.0417,0.9675,0.0690,0.7361,0.2410


In [20]:
selection_records_df = (
    pd.DataFrame(
        selection_records
    )
)

display(
    selection_records_df
)

,Fold,SelectedFeatures,NumberSelected,CFS_Merit
0,1,"[OverTime, StockOptionLevel, JobLevel, YearsAt...",4,0.064519
1,2,"[OverTime, JobLevel, StockOptionLevel, YearsWi...",11,0.066776
2,3,"[OverTime, JobLevel, StockOptionLevel, YearsWi...",9,0.066740
3,4,"[OverTime, StockOptionLevel, JobLevel, YearsAt...",10,0.062742
4,5,"[OverTime, StockOptionLevel, JobLevel, YearsAt...",9,0.064023
5,6,"[OverTime, JobLevel, StockOptionLevel, YearsAt...",9,0.065041
6,7,"[OverTime, JobLevel, StockOptionLevel, YearsAt...",9,0.061648
7,8,"[OverTime, JobLevel, StockOptionLevel, YearsAt...",4,0.066788
8,9,"[OverTime, JobLevel, StockOptionLevel, YearsAt...",10,0.064986
9,10,"[OverTime, StockOptionLevel, JobLevel]",3,0.068347


## 9. Cross-Validation Summary

The mean and standard deviation of the evaluation metrics are calculated across
the ten validation folds for every probabilistic classifier.

In [21]:
metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "AUC",
    "Brier"
]

summary_rows = []

for model_name in (
    multivariate_fold_results[
        "Model"
    ].unique()
):

    model_data = (
        multivariate_fold_results[
            multivariate_fold_results[
                "Model"
            ] == model_name
        ]
    )

    row = {
        "Model": model_name
    }

    for metric in metrics:

        row[
            f"{metric}_mean"
        ] = (
            model_data[
                metric
            ].mean()
        )

        row[
            f"{metric}_std"
        ] = (
            model_data[
                metric
            ].std()
        )

    summary_rows.append(
        row
    )

multivariate_results = (
    pd.DataFrame(
        summary_rows
    )
)

display(
    multivariate_results
    .round(4)
)

,Model,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,Specificity_mean,Specificity_std,F1_mean,F1_std,AUC_mean,AUC_std,Brier_mean,Brier_std
0,Mixed Naive Bayes,0.8388,0.0274,0.5157,0.1788,0.3531,0.1621,0.9318,0.0415,0.3962,0.1454,0.7616,0.0593,0.2386,0.0293
1,Logistic Regression,0.8551,0.0208,0.6471,0.2563,0.2014,0.1133,0.9805,0.0122,0.2997,0.1462,0.7692,0.0578,0.2229,0.0250
2,LDA,0.8544,0.0223,0.6246,0.2161,0.2192,0.1325,0.9765,0.0111,0.3134,0.1536,0.7428,0.0579,0.2335,0.0277
3,QDA,0.8395,0.0193,0.4934,0.1442,0.2525,0.1140,0.9521,0.0232,0.3259,0.1237,0.7204,0.0617,0.2463,0.0279


In [22]:
multivariate_summary = (
    multivariate_results[
        [
            "Model",
            "Accuracy_mean",
            "Precision_mean",
            "Recall_mean",
            "Specificity_mean",
            "F1_mean",
            "AUC_mean",
            "Brier_mean"
        ]
    ].copy()
)

multivariate_summary.columns = [
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "AUC",
    "Brier"
]

multivariate_summary = (
    multivariate_summary
    .sort_values(
        "F1",
        ascending=False
    )
)

display(
    multivariate_summary
    .round(4)
)

,Model,Accuracy,Precision,Recall,Specificity,F1,AUC,Brier
0,Mixed Naive Bayes,0.8388,0.5157,0.3531,0.9318,0.3962,0.7616,0.2386
3,QDA,0.8395,0.4934,0.2525,0.9521,0.3259,0.7204,0.2463
2,LDA,0.8544,0.6246,0.2192,0.9765,0.3134,0.7428,0.2335
1,Logistic Regression,0.8551,0.6471,0.2014,0.9805,0.2997,0.7692,0.2229


## 10. Selection Stability

Since CFS is fitted independently in every training fold, the selected subset
may vary.

Selection frequency is analysed to identify predictors that are consistently
retained by the multivariate criterion.

In [23]:
selection_frequency = {
    feature: 0
    for feature in X.columns
}

for record in selection_records:

    for feature in record[
        "SelectedFeatures"
    ]:

        selection_frequency[
            feature
        ] += 1

selection_frequency_df = (
    pd.DataFrame({
        "Feature":
            selection_frequency.keys(),

        "SelectedFolds":
            selection_frequency.values()
    })
    .sort_values(
        [
            "SelectedFolds",
            "Feature"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)

display(
    selection_frequency_df
)

,Feature,SelectedFolds
0,JobLevel,10
1,OverTime,10
2,StockOptionLevel,10
3,YearsAtCompany,9
4,Age,7
5,BusinessTravel,7
6,JobRole,7
7,MaritalStatus,7
8,JobInvolvement,6
9,YearsWithCurrManager,4


## 11. Comparison with the Full-Feature Baseline

The multivariate-filter results are compared with the complete 30-predictor
baseline.

Positive changes in F1, Recall, and AUC indicate improvement.

For Brier score, negative changes indicate improvement because lower values are
better.

In [24]:
baseline_summary = pd.read_csv(
    "../../results/pa2/baseline_summary.csv"
)

comparison = (
    baseline_summary[
        [
            "Model",
            "F1",
            "Recall",
            "AUC",
            "Brier"
        ]
    ]
    .merge(
        multivariate_summary[
            [
                "Model",
                "F1",
                "Recall",
                "AUC",
                "Brier"
            ]
        ],
        on="Model",
        suffixes=(
            "_Baseline",
            "_Multivariate"
        )
    )
)

comparison["F1_change"] = (
    comparison["F1_Multivariate"]
    - comparison["F1_Baseline"]
)

comparison["Recall_change"] = (
    comparison["Recall_Multivariate"]
    - comparison["Recall_Baseline"]
)

comparison["AUC_change"] = (
    comparison["AUC_Multivariate"]
    - comparison["AUC_Baseline"]
)

comparison["Brier_change"] = (
    comparison["Brier_Multivariate"]
    - comparison["Brier_Baseline"]
)

display(
    comparison.round(4)
)

,Model,F1_Baseline,Recall_Baseline,AUC_Baseline,Brier_Baseline,F1_Multivariate,Recall_Multivariate,AUC_Multivariate,Brier_Multivariate,F1_change,Recall_change,AUC_change,Brier_change
0,Logistic Regression,0.5373,0.4259,0.8375,0.1857,0.2997,0.2014,0.7692,0.2229,-0.2376,-0.2245,-0.0683,0.0372
1,Mixed Naive Bayes,0.5083,0.6453,0.7851,0.2902,0.3962,0.3531,0.7616,0.2386,-0.1121,-0.2922,-0.0235,-0.0516
2,LDA,0.4256,0.3168,0.8146,0.2036,0.3134,0.2192,0.7428,0.2335,-0.1122,-0.0976,-0.0719,0.0300
3,QDA,0.3946,0.3755,0.7211,0.2786,0.3259,0.2525,0.7204,0.2463,-0.0686,-0.1230,-0.0007,-0.0322


## 12. Comparison with Univariate Selection

The multivariate filter is also compared directly with the univariate filter.

This comparison shows whether considering redundancy and joint subset quality
provides an advantage over evaluating predictors independently.

In [25]:
univariate_summary = pd.read_csv(
    "../../results/pa2/univariate_summary.csv"
)

selection_comparison = (
    univariate_summary[
        [
            "Model",
            "F1",
            "Recall",
            "AUC",
            "Brier"
        ]
    ]
    .merge(
        multivariate_summary[
            [
                "Model",
                "F1",
                "Recall",
                "AUC",
                "Brier"
            ]
        ],
        on="Model",
        suffixes=(
            "_Univariate",
            "_Multivariate"
        )
    )
)

display(
    selection_comparison
    .round(4)
)

,Model,F1_Univariate,Recall_Univariate,AUC_Univariate,Brier_Univariate,F1_Multivariate,Recall_Multivariate,AUC_Multivariate,Brier_Multivariate
0,Mixed Naive Bayes,0.4600,0.6033,0.7633,0.3190,0.3962,0.3531,0.7616,0.2386
1,QDA,0.4240,0.3924,0.7302,0.2628,0.3259,0.2525,0.7204,0.2463
2,Logistic Regression,0.3736,0.2708,0.7919,0.2112,0.2997,0.2014,0.7692,0.2229
3,LDA,0.3700,0.2658,0.7714,0.2239,0.3134,0.2192,0.7428,0.2335


## 13. Export Results

The multivariate-filter performance, selected subsets, and selection
frequencies are stored for the final PA2 comparison.

In [26]:
multivariate_fold_results.to_csv(
    "../../results/pa2/multivariate_fold_results.csv",
    index=False
)

multivariate_results.to_csv(
    "../../results/pa2/multivariate_results.csv",
    index=False
)

multivariate_summary.to_csv(
    "../../results/pa2/multivariate_summary.csv",
    index=False
)

selection_records_df.to_csv(
    "../../results/pa2/multivariate_selected_subsets.csv",
    index=False
)

selection_frequency_df.to_csv(
    "../../results/pa2/multivariate_selection_frequency.csv",
    index=False
)

print(
    "PA2 multivariate results saved successfully."
)

PA2 multivariate results saved successfully.


## 14. Main Observations

The CFS multivariate filter selects relatively compact subsets of predictors,
with between 3 and 11 variables retained across the ten folds.

The descriptive full-data fit selects 9 variables:
`OverTime`, `JobLevel`, `StockOptionLevel`, `YearsAtCompany`, `Age`,
`BusinessTravel`, `JobRole`, `MaritalStatus`, and `JobInvolvement`.

Three predictors are selected in all ten folds:
`JobLevel`, `OverTime`, and `StockOptionLevel`.
`YearsAtCompany` is selected in nine folds, while `Age`, `BusinessTravel`,
`JobRole`, and `MaritalStatus` are each selected in seven folds.

Mixed Naive Bayes obtains the highest F1-measure under the multivariate
strategy (0.3962) and also the highest Recall (0.3531). However, both values
are lower than its full-feature baseline. Its AUC also decreases slightly,
although its Brier score improves from 0.2902 to 0.2386.

Logistic Regression is strongly affected by the multivariate reduction. Its
F1 decreases from 0.5373 to 0.2997, Recall from 0.4259 to 0.2014, and AUC from
0.8375 to 0.7692. Its Brier score also becomes worse.

LDA also performs worse than its baseline across F1, Recall, AUC, and Brier
score.

QDA shows a smaller degradation than the other classifiers. Its F1 decreases
from 0.3946 to 0.3259 and Recall from 0.3755 to 0.2525, while its AUC remains
almost unchanged. Its Brier score improves from 0.2786 to 0.2463.

Compared with the univariate filter, the multivariate CFS strategy produces
lower F1 and Recall for all four classifiers. It does, however, improve Brier
score for Mixed Naive Bayes and QDA relative to their univariate results.

Overall, the CFS filter identifies a compact and relatively stable subset of
predictors, but it does not improve the strongest predictive configuration.